# Preprocesamiento y pipelines primarios

Pipelines sin distinguir por cantidad de jets. Incluyen:
- Feature engineering (avg_jet_pt, log de masas y momentos)
- Imputacion de datos faltantes (-999 → NaN → mediana)
- Eliminacion de features muy correlacionadas (threshold 0.95)
- Eliminacion automatica de features originales post-transformacion
- StandardScaler / RobustScaler

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, StandardScaler

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from utils import (
    AvgJetPtTransformer,
    DropFeatures,
    DropHighlyCorrelatedFeatures,
    LogTransformFeatures,
    Modelo,
    basic_comb_train,
)

In [2]:
# Muestra chica para iterar rapido.
data = pd.read_csv(repo_root / "data" / "training.csv").sample(4000, random_state=42)

# Se deja Weight y EventId porque basic_comb_train usa Weight para AMS
# y saca Weight/EventId antes de entrenar.
X_train = data.drop(columns=["Label"]).replace(-999.0, np.nan)
Y_train = data["Label"]

print(f"Shape: {X_train.shape}")
print(f"NaN totales: {X_train.isna().sum().sum()}")

Shape: (4000, 32)
NaN totales: 25373


## Definicion de features a transformar

Features de masa y momento transverso reciben `log1p` porque sus distribuciones son fuertemente sesgadas a la derecha (tipico en fisica de particulas). `LogTransformFeatures` crea la columna `log_{nombre}` y **elimina la original** automaticamente.

In [3]:
# Features a las que se les aplica log1p (masas y momentos transversos).
# Son las distribuciones right-skewed identificadas en la exploracion.
LOG_FEATURES = [
    # Masas derivadas
    "DER_mass_MMC",
    "DER_mass_transverse_met_lep",
    "DER_mass_vis",
    "DER_mass_jet_jet",
    # Momentos transversos primarios
    "PRI_tau_pt",
    "PRI_lep_pt",
    "PRI_jet_leading_pt",
    "PRI_jet_subleading_pt",
    "PRI_jet_all_pt",
    "PRI_met",
    "PRI_met_sumet",
    # Sumas de pt derivadas
    "DER_sum_pt",
    "DER_pt_h",
    "DER_pt_tot",
]

## Pipelines

Orden de cada pipeline completo:
1. **AvgJetPtTransformer** — agrega `avg_jet_pt` (feature engineering)
2. **LogTransformFeatures** — aplica `log1p` a masas/pt, elimina originales
3. **DropHighlyCorrelatedFeatures** — elimina features con corr > 0.95 (aprende en fit)
4. **SimpleImputer** — rellena NaN restantes con la mediana
5. **StandardScaler / RobustScaler** — escalado final

Se incluye un pipeline baseline (solo imputacion + escalado) para comparar.

In [4]:
pipelines = [
    # --- Pipelines completos (feature engineering + limpieza + escalado) ---
    (
        "full_standard",
        Pipeline([
            ("avg_jet_pt", AvgJetPtTransformer()),
            ("log_transform", LogTransformFeatures(columns=LOG_FEATURES)),
            ("drop_correlated", DropHighlyCorrelatedFeatures(threshold=0.95)),
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]),
    ),
    (
        "full_robust",
        Pipeline([
            ("avg_jet_pt", AvgJetPtTransformer()),
            ("log_transform", LogTransformFeatures(columns=LOG_FEATURES)),
            ("drop_correlated", DropHighlyCorrelatedFeatures(threshold=0.95)),
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler()),
        ]),
    ),
    # --- Sin eliminacion de correlaciones (para ver si aporta o no) ---
    (
        "log_standard",
        Pipeline([
            ("avg_jet_pt", AvgJetPtTransformer()),
            ("log_transform", LogTransformFeatures(columns=LOG_FEATURES)),
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]),
    ),
    # --- Baseline: solo imputacion + escalado, sin feature engineering ---
    (
        "baseline_standard",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]),
    ),
]

## Modelos y entrenamiento

In [5]:
models = [
    Modelo("logreg", LogisticRegression(max_iter=1500, class_weight="balanced")),
    Modelo(
        "random_forest",
        RandomForestClassifier(
            n_estimators=120, random_state=42, n_jobs=-1, class_weight="balanced"
        ),
    ),
    Modelo("gradient_boosting", GradientBoostingClassifier(random_state=42)),
]

resultados = basic_comb_train(
    X_train=X_train,
    Y_train=Y_train,
    models=models,
    pipelines=pipelines,
    k_fold=3,
    path=repo_root / "modelos_prueba",
)

## Resultados

In [ ]:
resumen = pd.DataFrame([
    {
        "pipeline": r["pipeline_name"],
        "modelo": r["model_name"],
        "ams": round(r["mean_ams"], 4),
        "f1": round(r["mean_f1"], 4),
        "accuracy": round(r["mean_accuracy"], 4),
        "precision": round(r["mean_precision"], 4),
        "recall": round(r["mean_recall"], 4),
    }
    for r in resultados
])

resumen

# --- PCA sobre el mejor pipeline para visualizar separacion de clases ---
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Toma el pipeline del mejor resultado y transforma los datos.
best = resultados[0]
best_pipeline = best["model"].named_steps["preprocess"]
X_features = X_train.drop(columns=["Weight", "EventId"], errors="ignore")
X_transformed = best_pipeline.transform(X_features)

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_transformed)

y_binary = (Y_train == "s").values

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_pca[~y_binary, 0], X_pca[~y_binary, 1], alpha=0.3, s=5, label="background", c="tab:blue")
ax.scatter(X_pca[y_binary, 0], X_pca[y_binary, 1], alpha=0.3, s=5, label="signal", c="tab:orange")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} varianza)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} varianza)")
ax.set_title(f"PCA 2D — pipeline: {best['pipeline_name']}")
ax.legend(markerscale=4)
plt.tight_layout()
plt.show()

print(f"Varianza explicada PC1+PC2: {pca.explained_variance_ratio_[:2].sum():.1%}")